# MATH-F V1 Demo (Colab)

This notebook runs the MATH-F V1 pipeline **for real**: it installs a
Lean 4 toolchain and Mathlib (this requires network access, which is
why this step could not be done in the sandboxed environment that
built this repository — see the main `README.md` "Known Limitations"
section), builds the verification project, sanity-checks it, then runs
the actual experiment against the real `LeanVerifier`.

Steps:
1. Clone the repository
2. Install `elan` + the pinned Lean toolchain
3. Resolve and build the Lean project (fetch Mathlib)
4. Run a tiny Lean sanity check
5. Run the V1 development benchmark for real
6. Display aggregate metrics
7. Display at least one complete trajectory
8. Save results


## 1. Clone the repository

If you're running this notebook from inside an already-cloned copy of the repo (e.g. you uploaded the whole folder to Colab), this cell detects that and skips cloning.

In [ ]:
import os

REPO_URL = "https://github.com/YOUR_ORG/math-f.git"  # <-- replace with the real repo URL
REPO_DIR = "/content/math-f"

if os.path.isdir(os.path.join(REPO_DIR, "src", "mathf")):
    print(f"Found existing checkout at {REPO_DIR}, skipping clone.")
else:
    !git clone "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!ls


## 2. Install Python dependencies

In [ ]:
!pip install -q -r requirements.txt


## 3. Install elan (Lean toolchain manager) + Lean

This reads `lean_project/lean-toolchain` and installs exactly that
toolchain version. If Mathlib has since moved past the pinned version,
`lake update` in the next step will complain clearly — see
`lean_project/README.md` for how to update the pin.

In [ ]:
import os

if not os.path.exists(os.path.expanduser("~/.elan/bin/elan")):
    !curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh -s -- -y -q
os.environ["PATH"] = os.path.expanduser("~/.elan/bin") + ":" + os.environ["PATH"]
!elan --version
!cat lean_project/lean-toolchain


## 4. Resolve dependencies, fetch prebuilt Mathlib cache, and build

`lake exe cache get` downloads prebuilt `.olean` files for Mathlib instead of compiling it from source, which is the difference between minutes and hours.

In [ ]:
%cd /content/math-f/lean_project
!lake update
!lake exe cache get
!lake build
%cd /content/math-f


## 5. Sanity check: does a trivial proof actually verify?

This calls `LeanVerifier.check_environment()` directly, which is the same code
path `experiments/run_v1.py --check-lean` uses.

**Heads up on timing:** every verification call spawns a *fresh* `lean`
process, and `import Mathlib` forces that process to deserialize the entire
Mathlib environment from disk before it even looks at the one-line goal
below. Even with the prebuilt cache from `lake exe cache get`, this cold
import routinely takes well over a minute on shared/cloud disks (Colab
included) -- it is not a sign anything is broken. `timeout_seconds=180`
below reflects that; if it *still* times out, scroll up to the previous
cell's output and confirm `lake exe cache get` reported downloaded files
with no errors, and that `lake build` did not print `Building
Mathlib....` lines (that would mean the cache wasn't used and it's
compiling Mathlib from source, which takes hours -- not something a
larger timeout here will fix; re-run `lake exe cache get` instead).


In [ ]:
import sys
import time

sys.path.insert(0, "/content/math-f/src")

from mathf.verifiers.lean_verifier import LeanVerifier

verifier = LeanVerifier(lean_project_dir="/content/math-f/lean_project", timeout_seconds=180)

start = time.time()
result = verifier.check_environment()
elapsed = time.time() - start

print(f"verified: {result.verified}")
print(f"status: {result.status}")
print(f"elapsed: {elapsed:.1f}s")
if not result.verified:
    print("stdout:", result.raw_stdout)
    print("stderr:", result.raw_stderr)
    print("metadata:", result.verifier_metadata)
assert result.verified, (
    "Lean/Mathlib sanity check failed. If status is TIMEOUT, first re-check "
    "that 'lake exe cache get' succeeded in the previous cell (no errors, "
    "files downloaded) -- if it did and this still times out, try raising "
    "timeout_seconds further (e.g. 300-600) before concluding something is "
    "actually broken."
)
print(f"Lean + Mathlib environment OK ({elapsed:.1f}s for a cold Mathlib import).")


## 6. Run the V1 development benchmark for real

This uses the real `LeanVerifier` (no `--allow-mock`), against the 16-problem benchmark in `datasets/v1/problems.json`.

In [ ]:
!python experiments/run_v1.py --verifier lean --timeout-seconds 180


## 7. Display aggregate metrics

In [ ]:
import json
import glob

latest_result = max(glob.glob("results/experiment_*.json"), key=lambda p: __import__("os").path.getmtime(p))
with open(latest_result) as f:
    experiment = json.load(f)

print(f"Experiment: {experiment['reproducibility']['experiment_id']}")
print(f"Lean version: {experiment['reproducibility']['lean_version']}")
print(f"Verifier is mock: {experiment['reproducibility']['verifier_is_mock']}")
print()
print(json.dumps(experiment["metrics"], indent=2))


In [ ]:
# Optional: a small bar chart of cumulative success by attempt, if matplotlib is available.
import matplotlib.pyplot as plt

metrics = experiment["metrics"]
attempts = []
n = 1
while f"success_after_attempt_{n}" in metrics:
    attempts.append(metrics[f"success_after_attempt_{n}"])
    n += 1

plt.bar(range(1, len(attempts) + 1), attempts)
plt.xlabel("Attempt number")
plt.ylabel("Cumulative problems solved")
plt.title("Cumulative success by attempt")
plt.show()


## 8. Display at least one complete trajectory

One solved-with-retries trajectory and, if present, the permanently-unsolved one (`mathf_016`).

In [ ]:
import glob
import json

def load_trajectory_for(problem_id):
    matches = glob.glob(f"trajectories/{problem_id}_*.json")
    if not matches:
        return None
    with open(matches[0]) as f:
        return json.load(f)

for pid in ["mathf_003", "mathf_016"]:
    traj = load_trajectory_for(pid)
    if traj is None:
        print(f"No trajectory found for {pid} (did the run above complete?)")
        continue
    print(f"=== {pid} ===  solved={traj['solved']}  attempts_used={traj['attempts_used']}  final_status={traj['final_status']}")
    for a in traj["attempts"]:
        v = a["verification"]
        print(f"  attempt {a['attempt']}: proof={a['candidate']['proof']!r:20} verified={v['verified']!s:5} status={v['status']} duplicate={a['duplicate']}")
    print()


## 9. Failure analysis

In [ ]:
print(json.dumps(experiment["failure_analysis"], indent=2))


## 10. Save results

`trajectories/` and `results/` already contain everything from this run. If you want a single downloadable archive:

In [ ]:
import shutil

shutil.make_archive("/content/mathf_v1_results", "zip", root_dir="/content/math-f", base_dir="results")
shutil.make_archive("/content/mathf_v1_trajectories", "zip", root_dir="/content/math-f", base_dir="trajectories")
print("Saved /content/mathf_v1_results.zip and /content/mathf_v1_trajectories.zip")

try:
    from google.colab import files
    files.download("/content/mathf_v1_results.zip")
    files.download("/content/mathf_v1_trajectories.zip")
except ImportError:
    pass  # not running in Colab
